# JDMAgent — Démo interactive

> Agentification de **JeuxDeMots**, le graphe lexical du français (~2M nœuds,
> 180+ relations typées). Ce notebook te permet d'explorer le graphe,
> de vérifier des affirmations factuelles, et d'enrichir la base — **sans rien
> installer sur ta machine**, directement depuis Colab.

**Repo GitHub** : <https://github.com/expAg/JDMAgent>
**Documentation utilisateur** : voir [USAGE.md](https://github.com/expAg/JDMAgent/blob/main/USAGE.md)

---

## Pré-requis

Exécute la première cellule pour installer le package. ~30 secondes au premier lancement.


In [ ]:
# Installation depuis le repo GitHub (~30 s)
!pip install -q git+https://github.com/expAg/JDMAgent.git


---

## 1. Explorer JDM — sans LLM, instantané

On instancie le client typé et on requête des relations directes. Le cache
disque garantit qu'au 2ᵉ appel d'un même endpoint c'est instantané.


In [ ]:
from jdm_agent.client import JDMClient

c = JDMClient()

# Recherche un terme
n = c.node_by_name("chat")
print(f"Terme : {n.name} (id={n.id}, poids global w={n.w})")

# Synonymes filtrés par poids (helper de haut niveau)
print("\nSynonymes de 'voiture' :")
for s in c.synonyms("voiture", min_weight=50, limit=5):
    print(f"  - {s.name} (w={s.w})")

# Hyperonymes
print("\nHyperonymes de 'chat' :")
for h in c.hypernyms("chat", min_weight=200, limit=5):
    print(f"  - {h.name} (w={h.w})")


### Désambiguïsation des termes polysémiques

JeuxDeMots encode plusieurs sens d'un même mot avec des refinements
(`avocat>116477>66699` = avocat juriste). Notre client les **décode
automatiquement** en français lisible.


In [ ]:
senses = c.refinements_decoded("avocat")
for s in sorted(senses, key=lambda x: -x.weight)[:6]:
    print(f"  {s.decoded:<40s}  w={s.weight:.0f}  brut={s.name}")


### Relations entre deux termes

Quel est le rapport entre `chat` et `internet` selon JDM ?


In [ ]:
res = c.relations_between("chat", "internet", min_weight=20)
for r in sorted(res.relations, key=lambda x: -x.w)[:5]:
    rel = c.relation_type_name(r.type) or f"type_{r.type}"
    print(f"  chat | {rel} | internet  (w={r.w:.0f})")


---

## 2. Fact-checker — vérification déterministe

Le `verify_claim` regarde dans JDM si un triplet est supporté, contradicted
(via `r_isa-incompatible`), ou inconnu. **Aucun appel LLM** — c'est de la
logique sur le graphe.


In [ ]:
from jdm_agent.factcheck import Claim, verify_claim

claims = [
    Claim(text="sang rouge",      subject="sang",    relation="r_has_color", object="rouge"),
    Claim(text="baleine poisson", subject="baleine", relation="r_isa",       object="poisson"),
    Claim(text="chat mammifère",  subject="chat",    relation="r_isa",       object="mammifère"),
    Claim(text="saumon poisson",  subject="saumon",  relation="r_isa",       object="poisson"),
    Claim(text="saumon mammifère", subject="saumon", relation="r_isa",       object="mammifère"),
]

ICONS = {"supported": "✓", "contradicted": "✗", "unknown": "?"}
for cl in claims:
    v = verify_claim(c, cl)
    print(f"{ICONS[v.status.value]} [{v.status.value.upper():<13s}] "
          f"{cl.subject} | {cl.relation} | {cl.object}  (conf={v.confidence:.2f})")
    print(f"     → {v.explanation[:120]}")


**Observe** : `baleine r_isa poisson` est correctement détecté CONTRADICTED
parce que JDM contient `baleine r_isa mammifère` ET `mammifère
r_isa-incompatible poisson`. C'est une inférence à 2 sauts faite **sans LLM**.

---

## 3. Enrichissement actif — détection des trous

Pour un terme moderne (smartphone, drone, télétravail…), JDM peut avoir
des trous de couverture. On les détecte automatiquement.


In [ ]:
from jdm_agent.enrich import detect_gaps

gaps = detect_gaps(
    c, "drone",
    target_relations=["r_has_part", "r_telic_role", "r_carac"],
)

print(f"{len(gaps)} gap(s) identifié(s) pour 'drone' :\n")
for g in gaps:
    print(f"  [{g.gap_type.value:<14s}] {g.term} | {g.relation}")
    print(f"     {g.detail[:140]}")


### Validation d'un candidat proposé manuellement

On peut soumettre un triplet candidat au validateur — il vérifie s'il
existe déjà, si la cible est connue, et s'il n'y a pas de contradiction.


In [ ]:
from jdm_agent.enrich import Candidate, validate_candidate

candidat = Candidate(
    term="drone", relation="r_has_part", target="caméra",
    confidence=0.9, rationale="composant standard d'un drone", source="manuel",
)
v = validate_candidate(c, candidat)
print(f"Status : {v.validation_status}")
print(f"Note   : {v.validation_note}")


---
## 4. Visualisation d'un sous-graphe JDM (vis-network)

Le module `jdm_agent.viz` construit un sous-graphe centré sur un terme et le sérialise en HTML interactif. Par défaut : profondeur 2, 11 relations standards, top-K par |w|. Les **négations** (poids négatif) apparaissent en rouge.


In [ ]:
# Si tu as sauté la cellule 1 ou redémarré le runtime, on (ré)installe ici.
try:
    import jdm_agent.viz  # noqa: F401
except ImportError:
    !pip install -q git+https://github.com/expAg/JDMAgent.git

from jdm_agent.client import JDMClient
from jdm_agent.viz import build_subgraph
from IPython.display import HTML, display
from pathlib import Path

client = JDMClient()
res = build_subgraph(
    'plat asiatique',
    client=client,
    depth=2,
    top_k_per_relation=6,
    output='html',
    output_path='/tmp/plat_asiatique_subgraph.html',
)
print(res['stats'])

html = Path(res['html_path']).read_text(encoding='utf-8')
# Embed en iframe srcdoc pour isoler les scripts vis-network du notebook.
display(HTML(
    f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}" '
    f'style="width:100%;height:700px;border:1px solid #ccc;border-radius:8px;"></iframe>'
))


---

## 4. Agent conversationnel (optionnel — requiert une clé API Anthropic)

Si tu veux discuter en langage naturel avec un LLM qui ne répond qu'avec
des triplets JDM cités, mets ta clé Anthropic ci-dessous. Elle reste
**dans cette session Colab** — non sauvegardée, non loggée.


In [ ]:
# OPTIONNEL — décommenter et coller ta clé pour activer l'agent
# import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["LLM_PROVIDER"] = "anthropic"
# os.environ["LLM_MODEL"]    = "claude-haiku-4-5"
#
# from jdm_agent.tools.jdm_agent import build_jdm_agent, ask
# agent = build_jdm_agent(client=c)
#
# out = ask(agent, "Quels sont les sens du mot avocat selon JDM ?")
# print(out["answer"])


---

## Pour aller plus loin

- 📚 **[Documentation complète](https://github.com/expAg/JDMAgent)** — README, USAGE.md, DEVELOPMENT.md
- 🤖 **Brancher dans Claude Code/Desktop** — voir la section MCP de USAGE.md
- 🌐 **Démo web** — [HF Spaces](#) *(à venir)*
- 📖 **Taxonomie des 180+ relations JDM** — [relation_definitions.md](https://github.com/expAg/JDMAgent/blob/main/relation_definitions.md)

**Crédits** : JeuxDeMots (M. Lafourcade, LIRMM/CNRS) — <https://www.jeuxdemots.org>
